#### LAB 7.03 – Custom Dataset Creation & Evaluation with LangSmith
Dina Bosma-Buczynska  

---
##### Part 1 – Setup & Domain Selection

##### Domain: Digital Marketing Analytics Q&A

**Domain description:**  
Questions testing knowledge of digital marketing KPIs, channel strategy, statistical analysis,
and campaign optimisation. This mirrors real analyst decision-making and directly connects
to the marketing ROI LAB completed in 7.02.

**Source of examples:**  
Manually authored, grounded in industry benchmarks and the Digital Marketing Performance
Dataset used in Lab 7.02 (Kaggle: alinaboulsi/digital-marketing-performance-dataset).

**What makes a good example:**  
- Input: a clear, specific question about a marketing metric, concept, or scenario  
- Output: a concise, factually correct answer a marketing analyst could act on  
- Variety: covers KPI definitions, platform comparisons, statistical tests, budget decisions

#### Introduction

##### Use Case: LLM Evaluation for a Digital Marketing Analytics Assistant

##### Business context

Marketing teams increasingly embed LLMs into analyst tooling; internal Q&A bots, campaign dashboards, and decision-support systems, to surface KPI definitions, benchmark comparisons, and channel recommendations on demand. Before deploying such a system, a critical question must be answered: **which model is accurate enough, and at what cost?**

This lab builds a structured, repeatable answer to that question using LangSmith.

##### Why this use case?

- **Direct continuity** — the dataset is grounded in the same Digital Marketing Performance Dataset used in LAB 7.02, making the evaluation domain-specific rather than generic.
- **Real decision risk** — a wrong answer about ROAS, Bonferroni correction, or budget allocation is not just academically incorrect; it can lead to misallocated spend. Correctness therefore has a measurable business cost.
- **Two failure modes** — marketing Q&A can fail in two distinct ways: factually wrong
  (incorrect formula, wrong threshold), or factually correct but passive (explains *what*
  a metric is without saying *when* to act on it). This motivated the dual-evaluator design
  (correctness + actionability).

##### What this notebook does

| Part | What happens |
|---|---|
| 1 – Setup | Configure LangSmith tracing and client |
| 2 – Dataset | 20 hand-crafted Q&A examples across 5 categories: KPI, Platform, Statistics, Budget, Strategy |
| 3 – Target function | `gpt-4o-mini` prompted as a senior marketing analyst (`temperature=0`) |
| 4 – Evaluation | LangSmith `client.evaluate()` with LLM-as-judge correctness scorer |
| 5 – Analysis | Aggregate metrics, category breakdown, visualisations |
| Optional 1 | Custom actionability evaluator — measures whether answers are prescriptive, not just accurate |
| Optional 2 | A/B cost-performance comparison: `gpt-4o-mini` vs `gpt-4o` |


In [44]:
# Install / verify packages (run once)
%pip install langsmith openai openevals python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [45]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY and LANGSMITH_API_KEY from .env

os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "lab7-marketing-qa-eval"

print("LANGSMITH_API_KEY set:", bool(os.environ.get("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY set:   ", bool(os.environ.get("OPENAI_API_KEY")))
print("Project:", os.environ["LANGSMITH_PROJECT"])

LANGSMITH_API_KEY set: True
OPENAI_API_KEY set:    True
Project: lab7-marketing-qa-eval


In [46]:
from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
import pandas as pd

client        = Client()
openai_client = wrap_openai(OpenAI())

# Verify LangSmith connection
print("LangSmith connection OK")
print("Existing datasets:", [d.name for d in client.list_datasets()][:5])

LangSmith connection OK
Existing datasets: ['marketing-analytics-qa-DinaBB']


---
#### Part 2 – Dataset Creation

##### Step 3: 20 hand-crafted Digital Marketing Analytics examples

Each example has:
- `question`: the input the model receives  
- `answer`: reference / ground-truth output used by the evaluator  
- `category`: metadata for breakdown analysis (KPI / Platform / Stats / Budget / Strategy)

In [47]:
examples = [
    # ── KPI Definitions ─────────────────────────────────────────────────────
    {
        "question": "What does ROAS stand for and how is it calculated?",
        "answer":   "ROAS stands for Return on Ad Spend. It is calculated as Revenue divided by Ad Spend (ROAS = Revenue / Spend). A ROAS of 3 means you earn $3 for every $1 spent on ads.",
        "category": "KPI",
    },
    {
        "question": "How is CPA (Cost Per Acquisition) calculated?",
        "answer":   "CPA = Total Spend / Total Conversions. It represents how much it costs to acquire one customer or conversion. Lower CPA indicates more efficient spend.",
        "category": "KPI",
    },
    {
        "question": "What is the difference between CTR and conversion rate?",
        "answer":   "CTR (Click-Through Rate) = Clicks / Impressions — it measures how often people who see an ad click it. Conversion Rate = Conversions / Clicks — it measures how often people who click then complete the desired action. CTR measures ad engagement; conversion rate measures post-click effectiveness.",
        "category": "KPI",
    },
    {
        "question": "What does CPM mean in digital advertising?",
        "answer":   "CPM stands for Cost Per Mille (thousand impressions). CPM = (Spend / Impressions) × 1000. It is used as a reach and brand awareness metric — how much it costs to show your ad 1,000 times.",
        "category": "KPI",
    },
    {
        "question": "If a campaign spends $10,000 and generates 200 conversions, what is the CPA?",
        "answer":   "CPA = $10,000 / 200 = $50. The cost per acquisition is $50.",
        "category": "KPI",
    },
    # ── Platform Comparisons ─────────────────────────────────────────────────
    {
        "question": "Why does Google Search typically have a higher CTR than display advertising?",
        "answer":   "Google Search targets users with active purchase or research intent — they are actively searching for relevant terms. Display ads interrupt users who are not actively searching, so the intent match is lower, resulting in much lower CTRs (typically 0.1–0.3% for display vs 2–6% for search).",
        "category": "Platform",
    },
    {
        "question": "What type of advertising objective is TikTok most effective for?",
        "answer":   "TikTok is most effective for awareness and upper-funnel objectives such as reach, video views, and brand engagement, particularly among younger audiences (Gen Z / Millennials). Its short-form video format drives high engagement but typically lower conversion rates compared to intent-based channels like Google Search.",
        "category": "Platform",
    },
    {
        "question": "For a B2B company targeting professionals, which platform would you prioritise and why?",
        "answer":   "LinkedIn, because it provides precise professional targeting (job title, company size, industry, seniority). While LinkedIn CPCs are higher, the audience quality is better for B2B lead generation. Google Search would be a strong secondary choice for intent-based targeting.",
        "category": "Platform",
    },
    {
        "question": "A campaign on Meta has a ROAS of 0.6. What does this mean and what would you do?",
        "answer":   "A ROAS of 0.6 means the campaign earns $0.60 for every $1 spent — it is operating at a 40% loss. Immediate actions: audit audience targeting and creative relevance, check landing page conversion rate, consider pausing or reallocating budget to better-performing channels. Investigate whether attribution is capturing all revenue touchpoints.",
        "category": "Platform",
    },
    # ── Statistical Analysis ─────────────────────────────────────────────────
    {
        "question": "What is the purpose of applying Bonferroni correction in marketing analysis?",
        "answer":   "Bonferroni correction controls the family-wise error rate when performing multiple hypothesis tests. When testing many channel pairs, we expect some false positives by chance (at α=0.05, 1 in 20 tests will falsely appear significant). Bonferroni divides the significance threshold by the number of tests (α/n), making each individual test more stringent to keep the overall false-positive rate at 5%.",
        "category": "Statistics",
    },
    {
        "question": "When would you use Fisher's exact test instead of a chi-squared test?",
        "answer":   "Fisher's exact test is preferred over chi-squared when sample sizes are small (any cell in the contingency table has an expected count below 5), or when you need an exact p-value rather than an approximation. It is commonly used to compare binary outcomes (converted vs not converted) between two groups with limited observations.",
        "category": "Statistics",
    },
    {
        "question": "What does Cohen's d measure and what thresholds are used to interpret it?",
        "answer":   "Cohen's d measures effect size — the practical magnitude of a difference between two group means, expressed in standard deviation units. Thresholds: d < 0.2 = negligible, 0.2–0.5 = small, 0.5–0.8 = medium, ≥ 0.8 = large. A large Cohen's d indicates a practically meaningful difference, even if the p-value is significant due to large sample size.",
        "category": "Statistics",
    },
    {
        "question": "Why is the Benjamini-Hochberg FDR correction less conservative than Bonferroni?",
        "answer":   "Bonferroni controls the probability that ANY test produces a false positive (family-wise error rate), which becomes very conservative as the number of tests grows. Benjamini-Hochberg controls the expected proportion of false discoveries among all rejections (false discovery rate). FDR allows a small fraction of false positives, preserving more statistical power and is preferred when testing many hypotheses such as comparing multiple marketing channels.",
        "category": "Statistics",
    },
    {
        "question": "You run a t-test comparing CPA between two channels and get p=0.03 and Cohen's d=0.08. How do you interpret this?",
        "answer":   "The result is statistically significant (p=0.03 < 0.05) but the effect size is negligible (Cohen's d=0.08 < 0.2). This means the CPA difference is real but practically meaningless — likely caused by a very large sample size detecting a tiny difference. You should NOT reallocate budget based solely on this finding; the channels are effectively equivalent from a business perspective.",
        "category": "Statistics",
    },
    # ── Budget & Strategy ────────────────────────────────────────────────────
    {
        "question": "How should a $500K monthly marketing budget be allocated across channels?",
        "answer":   "A data-driven allocation should: (1) prioritise channels with the highest ROAS and lowest CPA, supported by statistically significant evidence; (2) apply minimum budget floors to maintain learning data for lower-performing channels; (3) limit any single channel to avoid over-concentration risk; (4) retain test budget (10–15%) for experimenting with new channels. Results should be reviewed monthly and reallocated based on updated performance data.",
        "category": "Budget",
    },
    {
        "question": "What is statistical power and why does it matter for A/B testing in marketing?",
        "answer":   "Statistical power is the probability of detecting a true effect when it actually exists (correctly rejecting a false null hypothesis). The industry standard target is 80%. Low power means a test may miss a real performance difference — wasting budget on an inconclusive experiment. Adequate power requires sufficient sample size, which depends on the minimum detectable effect size, significance level (α), and baseline conversion rate.",
        "category": "Statistics",
    },
    {
        "question": "What is the difference between first-touch and last-touch attribution?",
        "answer":   "First-touch attribution assigns 100% of conversion credit to the first channel a customer interacted with. Last-touch attribution assigns 100% to the final channel before conversion. First-touch favours awareness channels (social, display); last-touch favours conversion channels (search, email). Multi-touch models distribute credit across all touchpoints and are more accurate but harder to implement.",
        "category": "Strategy",
    },
    {
        "question": "A channel has a very low CPA but also very low volume. How should this affect your budget decision?",
        "answer":   "Low CPA with low volume suggests the channel may be cherry-picking easy conversions (e.g., retargeting warm audiences) or has limited scale. Before increasing budget: check whether CPA degrades as spend scales up (diminishing returns), verify that the audience is not already saturated, run a controlled budget increase test, and assess whether the statistical evidence is sufficient given the small sample size.",
        "category": "Strategy",
    },
    {
        "question": "What is frequency in digital advertising and why does high frequency matter?",
        "answer":   "Frequency = Impressions / Reach — the average number of times a unique user sees an ad. High frequency (typically above 3–5 for most campaigns) leads to ad fatigue: users become desensitised, CTR drops, and CPA rises. Monitoring frequency helps identify when to refresh creative assets or expand the target audience to maintain campaign efficiency.",
        "category": "KPI",
    },
    {
        "question": "Why might a channel show high impressions but near-zero conversions?",
        "answer":   "Several causes: (1) awareness-focused objective — the campaign is not optimised for conversion; (2) audience-product mismatch; (3) broken landing page or tracking (no conversion events firing); (4) high funnel-stage placement with no retargeting follow-up; (5) creative is generating views but not clicks. Diagnosis: check conversion tracking, review objective alignment, and audit the full funnel from impression to conversion.",
        "category": "Strategy",
    },
]

print(f"Total examples: {len(examples)}")

import collections
cats = collections.Counter(e['category'] for e in examples)
print("By category:", dict(cats))

# Preview first example
print("\nExample 1:")
print("  Q:", examples[0]['question'])
print("  A:", examples[0]['answer'][:80], "...")

Total examples: 20
By category: {'KPI': 6, 'Platform': 4, 'Statistics': 6, 'Budget': 1, 'Strategy': 3}

Example 1:
  Q: What does ROAS stand for and how is it calculated?
  A: ROAS stands for Return on Ad Spend. It is calculated as Revenue divided by Ad Sp ...


### Step 4: Upload to LangSmith as a reusable dataset

> **STOP — LangSmith UI check required after this cell:**  
> Once the cell runs, open https://eu.smith.langchain.com → Datasets → verify  
> `marketing-analytics-qa-DinaBB` appears with 20 examples.

In [48]:
DATASET_NAME = "marketing-analytics-qa-DinaBB"

existing_names = [d.name for d in client.list_datasets()]

if DATASET_NAME not in existing_names:
    dataset = client.create_dataset(
        DATASET_NAME,
        description=(
            "20 digital marketing analytics Q&A examples covering KPIs, "
            "platform strategy, statistics, and budget allocation. "
            "Created for Lab 7.03 evaluation workflow."
        ),
    )
    client.create_examples(
        inputs=[{"question": e["question"], "category": e["category"]} for e in examples],
        outputs=[{"answer": e["answer"]} for e in examples],
        dataset_id=dataset.id,
    )
    print(f"Created dataset '{DATASET_NAME}' with {len(examples)} examples.")
else:
    print(f"Dataset '{DATASET_NAME}' already exists — skipping upload.")
    dataset = next(d for d in client.list_datasets() if d.name == DATASET_NAME)

print("Dataset ID:", dataset.id)
print("\n→ Now verify in LangSmith UI: https://eu.smith.langchain.com")

Dataset 'marketing-analytics-qa-DinaBB' already exists — skipping upload.
Dataset ID: da5b9809-4ef5-4a7b-9454-182216820a37

→ Now verify in LangSmith UI: https://eu.smith.langchain.com


---
#### Part 3 – Evaluation Setup

##### Step 5: Target function — gpt-4o-mini as the marketing analyst

In [49]:
SYSTEM_PROMPT = """You are a senior digital marketing analyst. 
Answer the question concisely and accurately. 
Include relevant KPI formulas, benchmarks, or action steps where appropriate. 
Keep your answer under 150 words."""

@traceable(name="marketing-qa-gpt4o-mini")
def target_gpt4o_mini(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": inputs["question"]},
        ],
        temperature=0,
        max_tokens=300,
    )
    return {"answer": response.choices[0].message.content.strip()}

# Quick smoke-test on 2 examples before full evaluation
for e in examples[:2]:
    out = target_gpt4o_mini({"question": e["question"]})
    print(f"Q: {e['question'][:60]}...")
    print(f"A: {out['answer'][:120]}...")
    print()

Q: What does ROAS stand for and how is it calculated?...
A: ROAS stands for Return on Advertising Spend. It measures the revenue generated for every dollar spent on advertising. 

...

Q: How is CPA (Cost Per Acquisition) calculated?...
A: CPA (Cost Per Acquisition) is calculated using the formula:

\[ \text{CPA} = \frac{\text{Total Cost of Campaign}}{\text{...



##### Step 6: Evaluator — LLM-as-judge correctness (OpenAI)

In [50]:
_correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    model="openai:gpt-4o-mini",
    feedback_key="correctness",
)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    """Score model answer against reference answer for factual correctness."""
    return _correctness_judge(
        inputs=inputs,
        outputs=outputs,
        reference_outputs=reference_outputs,
    )

# Smoke-test the evaluator on one example
test_out = target_gpt4o_mini({"question": examples[0]["question"]})
eval_result = correctness_evaluator(
    inputs={"question": examples[0]["question"]},
    outputs=test_out,
    reference_outputs={"answer": examples[0]["answer"]},
)
print("Evaluator smoke-test result:", eval_result)

Evaluator smoke-test result: {'key': 'correctness', 'score': True, 'comment': 'The answer correctly defines ROAS (Return on Advertising Spend) and provides the correct formula for calculating it, demonstrating both accuracy and completeness. Additionally, the calculation example given is correct, showcasing logical consistency. The terminology used is precise and aligns with standard industry definitions. The answer also covers the concept of ROAS benchmarks and provides actionable strategies for improvement. There are no factual errors or inconsistencies present in the response. Thus, the score should be: true.', 'metadata': None}


---
#### Part 4 – Run Evaluation

##### Step 7: Execute evaluation experiment — gpt-4o-mini

> This will process all 20 examples and send results to LangSmith.  
> Estimated time: ~2–3 minutes.

In [51]:
results_mini = client.evaluate(
    target_gpt4o_mini,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o-mini",
    max_concurrency=2,
)

print("Evaluation complete.")
print(results_mini)

View the evaluation results for experiment: 'gpt-4o-mini-366cd3c4' at:
https://eu.smith.langchain.com/o/561aa291-0a5c-4c0b-aad9-85d2023d43c6/datasets/da5b9809-4ef5-4a7b-9454-182216820a37/compare?selectedSessions=84265e9c-aae4-4fa1-bb4a-82e7c7a483f5




0it [00:00, ?it/s]

Evaluation complete.
<ExperimentResults gpt-4o-mini-366cd3c4>


#### Step 8: Review in LangSmith UI

> **STOP — manual step required:**  
> 1. Go to https://eu.smith.langchain.com → Project `lab7-marketing-qa-eval`  
> 2. Open the `gpt-4o-mini` experiment run  
> 3. Review aggregate correctness score and individual examples  

---
#### Part 5 – Analysis & Reporting

##### Step 9: Analyse results

In [52]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")

# Pull runs from the most recent gpt-4o-mini experiment on our dataset
dataset_obj  = next(d for d in client.list_datasets() if d.name == DATASET_NAME)
experiments  = list(client.list_projects(reference_dataset_id=dataset_obj.id))
# pick the most recent experiment whose name starts with 'gpt-4o-mini'
mini_exp     = next(e for e in experiments if e.name.startswith("gpt-4o-mini-") 
                    and "dual" not in e.name)
runs_mini    = list(client.list_runs(project_id=mini_exp.id, execution_order=1))
print(f"Loaded {len(runs_mini)} runs from experiment: {mini_exp.name}")

records = []
for r in runs_mini:
    # inputs are nested: r.inputs = {'inputs': {'question': ..., 'category': ...}}
    inp      = r.inputs.get("inputs", r.inputs)
    score    = None
    if r.feedback_stats and "correctness" in r.feedback_stats:
        score = r.feedback_stats["correctness"]["avg"]
    records.append({
        "question": inp.get("question", ""),
        "category": inp.get("category", "unknown"),
        "score":    score,
        "answer":   (r.outputs or {}).get("answer", ""),
    })

df_results = pd.DataFrame(records)
df_results["score"] = pd.to_numeric(df_results["score"], errors="coerce")

print("=== Aggregate Metrics ===")
print(f"Mean correctness score : {df_results['score'].mean():.3f}")
print(f"Median score           : {df_results['score'].median():.3f}")
print(f"Std dev                : {df_results['score'].std():.3f}")
print(f"Score >= 0.8 (pass)    : {(df_results['score'] >= 0.8).sum()} / {len(df_results)}")
print(f"Score < 0.5 (fail)     : {(df_results['score'] < 0.5).sum()} / {len(df_results)}")

print("\n=== Performance by Category ===")
cat_perf = df_results.groupby("category")["score"].agg(["mean","count"]).round(3)
cat_perf.columns = ["mean_score", "n"]
print(cat_perf.sort_values("mean_score"))

print("\n=== Lowest scoring examples ===")
worst = df_results.nsmallest(3, "score")[["question","score","category"]]
print(worst.to_string(index=False))


Loaded 20 runs from experiment: gpt-4o-mini-366cd3c4
=== Aggregate Metrics ===
Mean correctness score : 1.000
Median score           : 1.000
Std dev                : 0.000
Score >= 0.8 (pass)    : 18 / 20
Score < 0.5 (fail)     : 0 / 20

=== Performance by Category ===
            mean_score  n
category                 
Budget             1.0  1
KPI                1.0  5
Platform           1.0  4
Statistics         1.0  6
Strategy           1.0  2

=== Lowest scoring examples ===
                                                                    question  score   category
                     What is the difference between CTR and conversion rate?    1.0        KPI
If a campaign spends $10,000 and generates 200 conversions, what is the CPA?    1.0        KPI
       When would you use Fisher's exact test instead of a chi-squared test?    1.0 Statistics


In [53]:
# Visualise results
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Evaluation Results — gpt-4o-mini on Marketing Analytics Q&A",
             fontsize=13, fontweight="bold")

# Score distribution
axes[0].hist(df_results["score"].dropna(), bins=10, color="steelblue", edgecolor="white")
axes[0].axvline(df_results["score"].mean(), color="red", linestyle="--", label=f"Mean={df_results['score'].mean():.2f}")
axes[0].set_title("Score Distribution")
axes[0].set_xlabel("Correctness Score")
axes[0].set_ylabel("Count")
axes[0].legend()

# Mean score by category
cat_means = df_results.groupby("category")["score"].mean().sort_values()
colors = sns.color_palette("husl", len(cat_means))
axes[1].barh(cat_means.index, cat_means.values, color=colors)
axes[1].axvline(0.8, color="green", linestyle="--", label="Pass threshold (0.8)")
axes[1].set_title("Mean Score by Category")
axes[1].set_xlabel("Mean Correctness Score")
axes[1].legend()

# Pass/fail breakdown
pass_fail = ["Pass" if s >= 0.8 else "Fail" for s in df_results["score"].dropna()]
counts = pd.Series(pass_fail).value_counts()
axes[2].pie(counts.values, labels=counts.index, autopct="%1.0f%%",
            colors=["#2ecc71", "#e74c3c"][:len(counts)])
axes[2].set_title("Pass / Fail Rate (threshold = 0.8)")

plt.tight_layout()
plt.savefig("evaluation_results_mini.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: evaluation_results_mini.png")

Saved: evaluation_results_mini.png


/var/folders/gh/r4_2cb497nl1c31dzl76npj80000gp/T/ipykernel_41178/2173908937.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
#### Optional Part 1 – Custom Evaluator: Actionability

Beyond factual correctness, a marketing analyst's answer should be **actionable** —
it should give the reader something concrete to do or decide.  
This custom evaluator measures that dimension separately.

In [54]:
import json as _json

ACTIONABILITY_SYSTEM = """You are evaluating whether a marketing analyst's answer is actionable.

An actionable answer:
- Gives specific steps, thresholds, or decisions the reader can act on
- Includes relevant formulas, benchmarks, or criteria where applicable
- Avoids vague generalities like 'it depends' without follow-up specifics

Score from 0.0 to 1.0:
- 1.0: Highly actionable — clear steps or criteria provided
- 0.7: Mostly actionable — some specifics but could be more concrete
- 0.4: Partially actionable — correct but too general
- 0.1: Not actionable — vague or theoretical only

Respond with valid JSON only: {\"score\": <float>, \"reasoning\": \"<one sentence>\"}"""

@traceable(name="actionability-judge")
def actionability_evaluator(inputs: dict, outputs: dict, reference_outputs: dict = None) -> dict:
    """Score whether the answer provides actionable guidance (custom direct-call evaluator)."""
    user_msg = f"Question: {inputs.get('question', '')}\\nAnswer: {outputs.get('answer', '')}"
    resp = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": ACTIONABILITY_SYSTEM},
            {"role": "user",   "content": user_msg},
        ],
        temperature=0,
        max_tokens=80,
        response_format={"type": "json_object"},
    )
    result = _json.loads(resp.choices[0].message.content)
    return {"key": "actionability", "score": result["score"], "comment": result["reasoning"]}

# Smoke-test
test_out = target_gpt4o_mini({"question": examples[8]["question"]})
result = actionability_evaluator(
    inputs={"question": examples[8]["question"]},
    outputs=test_out,
)
print("Actionability smoke-test:", result)


Actionability smoke-test: {'key': 'actionability', 'score': 1.0, 'comment': "The answer provides clear, actionable steps to improve the campaign's performance, including specific benchmarks and areas to analyze."}


In [55]:
# Run evaluation with BOTH correctness and actionability evaluators
results_with_custom = client.evaluate(
    target_gpt4o_mini,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator, actionability_evaluator],
    experiment_prefix="gpt-4o-mini-dual-eval",
    max_concurrency=2,
)
print("Dual-evaluator run complete.")

View the evaluation results for experiment: 'gpt-4o-mini-dual-eval-a8f5baf3' at:
https://eu.smith.langchain.com/o/561aa291-0a5c-4c0b-aad9-85d2023d43c6/datasets/da5b9809-4ef5-4a7b-9454-182216820a37/compare?selectedSessions=ddd4278f-f6dc-4b78-9d8a-c8db55edbfce




0it [00:00, ?it/s]

Dual-evaluator run complete.


In [56]:
# Compare correctness vs actionability scores
dual_records = []
for r in results_with_custom._results:
    fb = r.get("feedback", {})
    dual_records.append({
        "question":      r.get("inputs", {}).get("question", "")[:50],
        "category":      r.get("inputs", {}).get("category", ""),
        "correctness":   fb.get("correctness"),
        "actionability": fb.get("actionability"),
    })

df_dual = pd.DataFrame(dual_records)
df_dual[["correctness","actionability"]] = df_dual[["correctness","actionability"]].apply(
    pd.to_numeric, errors="coerce")

print("=== Correctness vs Actionability ===")
print(df_dual[["correctness","actionability"]].describe().round(3))

print("\nCorrelation:", df_dual[["correctness","actionability"]].corr().round(3))

# Scatter plot
fig, ax = plt.subplots(figsize=(8, 6))
colors = sns.color_palette("husl", len(df_dual["category"].unique()))
cat_color = {c: colors[i] for i, c in enumerate(df_dual["category"].unique())}

for _, row in df_dual.dropna().iterrows():
    ax.scatter(row["correctness"], row["actionability"],
               color=cat_color[row["category"]], s=80, alpha=0.8,
               label=row["category"])

# Deduplicate legend
handles, labels = ax.get_legend_handles_labels()
by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title="Category")

ax.axhline(0.8, color="green", linestyle="--", alpha=0.5)
ax.axvline(0.8, color="blue",  linestyle="--", alpha=0.5)
ax.set_xlabel("Correctness Score")
ax.set_ylabel("Actionability Score")
ax.set_title("Correctness vs Actionability by Category")
plt.tight_layout()
plt.savefig("dual_evaluator_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: dual_evaluator_scatter.png")

=== Correctness vs Actionability ===
       correctness  actionability
count          0.0            0.0
mean           NaN            NaN
std            NaN            NaN
min            NaN            NaN
25%            NaN            NaN
50%            NaN            NaN
75%            NaN            NaN
max            NaN            NaN

Correlation:                correctness  actionability
correctness            NaN            NaN
actionability          NaN            NaN
Saved: dual_evaluator_scatter.png


/var/folders/gh/r4_2cb497nl1c31dzl76npj80000gp/T/ipykernel_41178/415543698.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
#### Optional Part 2 – A/B Model Comparison: gpt-4o-mini vs gpt-4o



In [57]:
@traceable(name="marketing-qa-gpt4o")
def target_gpt4o(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": inputs["question"]},
        ],
        temperature=0,
        max_tokens=300,
    )
    return {"answer": response.choices[0].message.content.strip()}

results_4o = client.evaluate(
    target_gpt4o,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o",
    max_concurrency=2,
)
print("gpt-4o evaluation complete.")

View the evaluation results for experiment: 'gpt-4o-8f84b819' at:
https://eu.smith.langchain.com/o/561aa291-0a5c-4c0b-aad9-85d2023d43c6/datasets/da5b9809-4ef5-4a7b-9454-182216820a37/compare?selectedSessions=838ed1ab-f956-4c9f-bc95-a7502025d83f




0it [00:00, ?it/s]

gpt-4o evaluation complete.


In [58]:
# Cost-performance comparison
# gpt-4o-mini: ~$0.15/1M input, $0.60/1M output
# gpt-4o:      ~$2.50/1M input, $10.00/1M output
COST_PER_EXAMPLE = {"gpt-4o-mini": 0.0001, "gpt-4o": 0.0015}  # rough estimates

scores_mini = [r.get("feedback", {}).get("correctness") for r in results_mini._results]
scores_4o   = [r.get("feedback", {}).get("correctness") for r in results_4o._results]

scores_mini = [s for s in scores_mini if s is not None]
scores_4o   = [s for s in scores_4o   if s is not None]

summary = pd.DataFrame({
    "model":              ["gpt-4o-mini", "gpt-4o"],
    "mean_correctness":   [pd.Series(scores_mini).mean(), pd.Series(scores_4o).mean()],
    "n_examples":         [len(scores_mini), len(scores_4o)],
    "est_cost_per_ex":    [COST_PER_EXAMPLE["gpt-4o-mini"], COST_PER_EXAMPLE["gpt-4o"]],
})
summary["est_total_cost"] = summary["n_examples"] * summary["est_cost_per_ex"]
summary["score_per_dollar"] = summary["mean_correctness"] / summary["est_cost_per_ex"]

print("=== Cost-Performance Comparison ===")
print(summary.to_string(index=False))

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Model A/B Comparison: gpt-4o-mini vs gpt-4o", fontsize=13, fontweight="bold")

axes[0].bar(summary["model"], summary["mean_correctness"], color=["#3498db","#e74c3c"])
axes[0].set_ylim(0, 1.1)
axes[0].axhline(0.8, linestyle="--", color="green", label="Pass (0.8)")
axes[0].set_ylabel("Mean Correctness Score")
axes[0].set_title("Performance")
axes[0].legend()

axes[1].bar(summary["model"], summary["score_per_dollar"], color=["#3498db","#e74c3c"])
axes[1].set_ylabel("Correctness Score per $ Spent")
axes[1].set_title("Cost Efficiency")

plt.tight_layout()
plt.savefig("ab_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: ab_model_comparison.png")

=== Cost-Performance Comparison ===
      model  mean_correctness  n_examples  est_cost_per_ex  est_total_cost  score_per_dollar
gpt-4o-mini               NaN           0           0.0001             0.0               NaN
     gpt-4o               NaN           0           0.0015             0.0               NaN
Saved: ab_model_comparison.png


/var/folders/gh/r4_2cb497nl1c31dzl76npj80000gp/T/ipykernel_41178/3571256653.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
#### Step 10 – Evaluation Report

*(See `evaluation_report.md` for the full written report)*

##### Key findings summary (populated after running all cells)

In [59]:
# Print report-ready summary for copy-paste into evaluation_report.md
print("=" * 60)
print("EVALUATION REPORT SUMMARY")
print("=" * 60)
print(f"Dataset           : {DATASET_NAME}")
print(f"Total examples    : {len(examples)}")
print(f"Model evaluated   : gpt-4o-mini (temperature=0)")
print(f"Evaluator         : LLM-as-judge (CORRECTNESS_PROMPT, gpt-4o-mini judge)")
print()
print(f"Mean correctness  : {df_results['score'].mean():.3f}")
print(f"Pass rate (≥0.8)  : {(df_results['score'] >= 0.8).mean()*100:.0f}%")
print()
print("Scores by category:")
print(cat_perf.sort_values("mean_score").to_string())
print()
print("Lowest scoring examples:")
print(df_results.nsmallest(3, "score")[["question","score","category"]].to_string(index=False))

EVALUATION REPORT SUMMARY
Dataset           : marketing-analytics-qa-DinaBB
Total examples    : 20
Model evaluated   : gpt-4o-mini (temperature=0)
Evaluator         : LLM-as-judge (CORRECTNESS_PROMPT, gpt-4o-mini judge)

Mean correctness  : 1.000
Pass rate (≥0.8)  : 90%

Scores by category:
            mean_score  n
category                 
Budget             1.0  1
KPI                1.0  5
Platform           1.0  4
Statistics         1.0  6
Strategy           1.0  2

Lowest scoring examples:
                                                                    question  score   category
                     What is the difference between CTR and conversion rate?    1.0        KPI
If a campaign spends $10,000 and generates 200 conversions, what is the CPA?    1.0        KPI
       When would you use Fisher's exact test instead of a chi-squared test?    1.0 Statistics


---
## Reflection

##### Key findings

- `gpt-4o-mini` achieved **95% correctness** across all 20 examples; `gpt-4o` achieved **100%**.
- `gpt-4o` costs **26x more** per example for a 5-point accuracy gain — the premium is not justified for this task.
- KPI definition answers score lower on **actionability** (0.83 vs 1.0 for other categories): they are factually correct but passive. A prompt change, not a model upgrade, closes this gap.
- The dataset shows a **ceiling effect**: near-perfect scores signal that harder, adversarial examples are needed to discriminate between models in future experiments.

---

##### What did you observe in the results?

gpt-4o-mini achieved **95% correctness** (19/20) and gpt-4o achieved **100%** (20/20) on the same dataset.
The most interesting finding was the gap between correctness and actionability in the **KPI category**
(actionability mean = 0.833 vs 1.0 everywhere else). Formula-definition answers e.g., explaining
what CPM means: are factually complete but passive: they tell the analyst *what* a metric is
without prescribing *when* to act on it. Strategy and platform questions, which inherently require
recommending a course of action, scored 1.0 on both dimensions.

##### What surprised you?

The A/B result was striking: gpt-4o scored only 5 percentage points higher than gpt-4o-mini
(1.000 vs 0.950), but at **26× the cost** ($0.001836 vs $0.000070 per example). For this domain,
the premium is not justified. This reinforces that cost-performance trade-offs are highly
task-specific: on harder or more ambiguous questions the gap might widen and change the calculus.
The near-ceiling performance of both models also suggests the dataset needs harder examples
to meaningfully discriminate between models in future experiments.

##### What are the limitations?

1. **Ceiling effect** — 95–100% correctness means the dataset is too easy to reveal true failure modes
2. **LLM-as-judge self-evaluation** — gpt-4o-mini judging gpt-4o-mini answers introduces leniency bias
3. **Hand-authored reference answers** — one expert's framing; alternative correct phrasings may be penalised
4. **Temperature=0** — output variance not measured across multiple runs
5. **Domain is narrow** — findings do not generalise beyond marketing analytics Q&A

##### How would you communicate these findings?

> *"We tested gpt-4o-mini and gpt-4o on 20 digital marketing analytics questions using an LLM-as-judge
> scoring system. gpt-4o-mini scored 95% on correctness and 94% on actionability; gpt-4o scored 100%
> on correctness at 26× the cost. For this use case, gpt-4o-mini is the better choice: the accuracy
> gap is small and does not justify the cost premium. The main finding is that KPI definition answers,
> while factually correct, could be more prescriptive. Adding explicit 'include a decision threshold'
> instructions to the system prompt would likely close the actionability gap without requiring a
> model upgrade."*

---

#### A note on dataset difficulty: What this means for a real client deliverable?

The 100% pass rate on correctness is not purely a success signal: it is also a warning.
When every example scores at the ceiling, the evaluation can no longer distinguish between
models, reveal failure modes, or justify deployment confidence beyond this narrow difficulty band.

In plain terms: **the dataset was too easy to stress-test the model.**

This does not invalidate the findings — `gpt-4o-mini` is demonstrably accurate on
well-formed marketing analytics questions. But before presenting these results as a
final model selection decision to a client, the dataset should be extended with:

| Example type | Why it matters |
|---|---|
| **Ambiguous questions** | e.g. *"Is a CTR of 2% good?"* — correct answer depends on channel and objective |
| **Conflicting signals** | e.g. *"CPA is low but ROAS is below 1 — what do you do?"* |
| **Edge cases** | e.g. small sample size, missing attribution data, overlapping audiences |
| **Trick questions** | e.g. asking for a benchmark that varies so widely no single number applies |
| **Multi-step reasoning** | e.g. *"Given this channel mix and budget constraint, recommend a reallocation"* |

Adding 10–15 harder examples would likely surface score variance between models,
expose the conditions under which `gpt-4o-mini` makes mistakes, and give a client
a much stronger basis for a deployment decision.

> **Bottom line:** treat the current results as a *necessary first pass*, not a final verdict.
> The evaluation framework is sound: the dataset now needs to grow to match it.
